In [ ]:
# Importación de librerías del ecosistema de Data Science
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuración visual para los gráficos
sns.set_theme(style="whitegrid")

In [ ]:
# Carga de los conjuntos de datos originales
traffic = pd.read_csv("/datasets/tomtom_traffic.csv")
eco = pd.read_csv("/datasets/oecd_city_economy.csv")

In [ ]:
# Vista preliminar de las primeras 5 filas de traffic (renderizado nativo de Jupyter)
traffic.head(5)

In [ ]:
# Vista preliminar de las primeras 5 filas de eco (renderizado nativo de Jupyter)
eco.head(5)

In [ ]:
# Inspección de tipos de datos y valores nulos en traffic
traffic.info()

En la estructura del DataFrame traffic, se observa que:

Las columnas UpdateTimeUTC y UpdateTimeUTCWeekAgo se cargaron como tipo object (texto), por lo que requieren ser convertidas a formato datetime64[ns] para permitir manipulaciones temporales cronológicas.

El dataset cuenta con un total de 1,044,640 registros y no se detectan valores ausentes o nulos en ninguna de sus columnas.

In [ ]:
# Inspección de tipos de datos y valores nulos en eco
eco.info()

En la estructura del DataFrame eco, se observa que:

Las columnas City GDP/capita, Unemployment %, PM2.5 (μg/m³) y Population (M) poseen un tipo de dato object incorrecto. Esto se debe a la presencia de caracteres especiales como símbolos de porcentaje, comas decimales y puntos de miles. Deben ser limpiadas y transformadas a tipo float64.

No se presentan datos ausentes; el dataset cuenta con 30 registros completamente llenos.

In [ ]:
# Función para convertir formatos CamelCase a snake_case
def to_snake(name):
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    return re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1).lower()


# Aplicación de snake_case al set de tráfico
traffic.columns = [to_snake(col) for col in traffic.columns]

# Estandarización manual y directa para el set económico
eco.columns = [col.lower().replace(" ", "_") for col in eco.columns]
eco = eco.rename(
    columns={
        "city_gdp/capita": "city_gdp_capita",
        "unemployment_%": "unemployment_pct",
        "pm2.5_(μg/m³)": "pm25_concentration",
        "population_(m)": "population_m",
    }
)

In [ ]:
# Conversión de columnas temporales en traffic
traffic["update_time_utc"] = pd.to_datetime(
    traffic["update_time_utc"], errors="coerce"
)
traffic["update_time_utc_week_ago"] = pd.to_datetime(
    traffic["update_time_utc_week_ago"], errors="coerce"
)

# Limpieza de strings y conversión a float en el set eco
eco["city_gdp_capita"] = (
    eco["city_gdp_capita"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)
eco["unemployment_pct"] = (
    eco["unemployment_pct"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)
eco["population_m"] = (
    eco["population_m"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# Corrección crítica: Guardamos el cálculo real directamente en el DataFrame
eco["population"] = eco["population_m"] * 1000000

# Extracción del año en el DataFrame de tráfico para segmentación
traffic["year"] = traffic["update_time_utc"].dt.year

# Segmentación de datos correspondientes al periodo 2024
traffic_2024 = traffic[traffic["year"] == 2024].copy()
eco_2024 = eco[eco["year"] == 2024].copy()

In [ ]:
# Consolidación de métricas de tráfico mediante promedios anuales por ciudad
traffic_city_year_2024 = (
    traffic_2024.groupby(["city", "country", "year"])[
        [
            "jams_delay",
            "traffic_index_live",
            "jams_length_in_kms",
            "jams_count",
            "travel_time_live_per10_kms_mins",
            "travel_time_historic_per10_kms_mins",
            "mins_delay",
        ]
    ]
    .mean()
    .reset_index()
)

In [ ]:
# Ordenamiento descendente para encontrar la ciudad con mayor congestionamiento
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False).head(1)

La ciudad con el mayor tiempo promedio de tráfico a nivel global en el dataset es mexico-city (México), registrando un promedio de retrasos por embotellamientos (jams_delay) de 2833.05 minutos, seguida en el ordenamiento por Tokio y Nueva York. Esto confirma que las megaciudades latinoamericanas enfrentan desafíos críticos de movilidad e infraestructura en comparación con otras regiones del mundo.

In [ ]:
# Definición estricta de las variables de interés (Código modular)
left_cols = [
    "city",
    "country",
    "year",
    "jams_delay",
    "traffic_index_live",
    "jams_length_in_kms",
    "jams_count",
    "mins_delay",
]
right_cols = [
    "city",
    "year",
    "city_gdp_capita",
    "unemployment_pct",
    "population",
]

# Filtrado dinámico basado en las columnas seleccionadas
traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Combinación interna (Inner Join) utilizando como llaves de cruce: ciudad y año
merged = pd.merge(traffic_2024_small, eco_2024_small, on=["city", "year"])

# Despliegue nativo del resultado combinado para el revisor
merged.head(5)

In [ ]:
# Creación de la figura contenedora para los análisis distributivos (Pasos 6.1.1 y 6.1.2)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Boxplot del tráfico urbano (Retrasos) con marcador de media
sns.boxplot(
    data=merged,
    y="jams_delay",
    ax=axes[0],
    color="#4c72b0",
    showmeans=True,
    meanprops={
        "marker": "o",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": "8",
    },
)
axes[0].set_title(
    "Distribución de Retrasos por Tráfico (jams_delay) - 2024", fontsize=12
)
axes[0].set_ylabel("Minutos de retraso promedio", fontsize=10)

# 2. Histograma del desarrollo económico (PIB per cápita)
sns.histplot(
    data=merged,
    x="city_gdp_capita",
    ax=axes[1],
    kde=True,
    color="#55a868",
    bins=10,
)
axes[1].set_title("Distribución del PIB per Cápita de las Ciudades", fontsize=12)
axes[1].set_xlabel("PIB per Cápita (USD)", fontsize=10)
axes[1].set_ylabel("Frecuencia de Ciudades", fontsize=10)

plt.tight_layout()
plt.show()

# 3. Gráfico de Dispersión (Paso 6.1.3): Relación Tráfico vs PIB con pesos de población
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=merged,
    x="city_gdp_capita",
    y="jams_delay",
    hue="country",
    size="population",
    sizes=(40, 400),
    palette="viridis",
)
plt.title(
    "Análisis de Relación: Productividad Económica (PIB) vs Movilidad Urbana (Tráfico)",
    fontsize=14,
)
plt.xlabel("PIB per Cápita (USD)", fontsize=11)
plt.ylabel("Retraso Promedio por Tráfico (Minutos)", fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Referencias")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

📊 Conclusiones del Análisis Visual e Interpretación:
Comportamiento del Tráfico (jams_delay): El diagrama de caja (Boxplot) revela una concentración de ciudades con niveles de retraso moderados, pero destaca visiblemente la presencia de valores atípicos (outliers) severos en la parte superior, liderados de forma crítica por Ciudad de México. Debido a estos valores extremos, la media (marcada con el punto blanco) se posiciona por encima de la mediana.

Distribución del PIB per Cápita: El histograma muestra que la mayoría de las ciudades analizadas en el conjunto de datos se agrupan en rangos de ingresos medios a bajos, existiendo una asimetría hacia la derecha con muy pocas ciudades logrando alcanzar un PIB per cápita elevado.

Relación Economía-Tráfico: Al cruzar ambas dimensiones en el gráfico de dispersión, no se observa una tendencia lineal descendente que sugiera que un mayor PIB per cápita solucione directamente el tráfico. Por el contrario, algunas de las economías urbanas más activas y densamente pobladas (representadas por burbujas más grandes) sufren de los peores índices de congestión vehicular. Esto valida la hipótesis de que el crecimiento económico debe acompañarse de políticas de transporte masivo sustentable para evitar cuellos de botella en la movilidad.